In [1]:
# !pip install pyxdf
import os
import pyxdf
import numpy as np
import json

In [2]:
def get_footer_info(stream):

    start_time = float(stream['footer']['info']['first_timestamp'][0])
    end_time = float(stream['footer']['info']['last_timestamp'][0])
    sample_count = int(stream['footer']['info']['sample_count'][0])
    duration = round(end_time - start_time)
    sampling_rate = round(sample_count/ duration)

    return start_time, end_time, sample_count, duration, sampling_rate

In [3]:
def convert_ndarray_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, list):
        return [convert_ndarray_to_list(x) for x in obj]
    if isinstance(obj, dict):
        return {k: convert_ndarray_to_list(v) for k, v in obj.items()}
    return obj

In [4]:
def get_physio_data_tobbi_gel(file_path):

    streams, file_header = pyxdf.load_xdf(file_path, verbose=False)

    # Initialize variables
    gaze2D, gaze3D, pupil, gazeOrigin, gazeDirection = [], [], [], [], []
    gaze2D_timestamps, gaze3D_timestamps, pupil_timestamps, gazeOrigin_timestamps, gazeDirection_timestamps = [], [], [], [], []
    gaze2D_sampling_rate, gaze3D_sampling_rate, pupil_sampling_rate, gazeOrigin_sampling_rate, gazeDirection_sampling_rate = 0, 0, 0, 0, 0

    marker, eeg = [], []
    marker_timestamps, eeg_timestamps = [], []
    marker_sampling_rate, eeg_sampling_rate = 0, 0

    eeg_channels = []

    for i, stream in enumerate(streams):

        stream_name = stream['info']['name'][0]

        match stream_name:

            case 'Gaze2d':

                gaze2D = stream['time_series']

                if isinstance(gaze2D, np.ndarray):
                    gaze2D = gaze2D.tolist()

                if gaze2D != []:

                    gaze2D_timestamps = stream['time_stamps']

                    if isinstance(gaze2D_timestamps, np.ndarray):
                        gaze2D_timestamps = gaze2D_timestamps.tolist()

                    _, _, _, _, gaze2D_sampling_rate = get_footer_info(stream)


            case 'Explore_84A1_Marker': # Marker from Mentalab

                marker = stream['time_series']

                if isinstance(marker, np.ndarray):
                    marker = marker.tolist()

                if marker != []:

                    marker_timestamps = stream['time_stamps']

                    if isinstance(marker_timestamps, np.ndarray):
                        marker_timestamps = marker_timestamps.tolist()

                    _, _, _, _, marker_sampling_rate = get_footer_info(stream)

            case 'Gaze3d':

                gaze3D = stream['time_series']

                if isinstance(gaze3D, np.ndarray):
                    gaze3D = gaze3D.tolist()

                if gaze3D != []:

                    gaze3D_timestamps = stream['time_stamps']

                    if isinstance(gaze3D_timestamps, np.ndarray):
                        gaze3D_timestamps = gaze3D_timestamps.tolist()

                    _, _, _, _, gaze3D_sampling_rate = get_footer_info(stream)

            case 'Pupil':

                pupil = stream['time_series']

                if isinstance(pupil, np.ndarray):
                    pupil = pupil.tolist()

                if pupil != []:

                    pupil_timestamps = stream['time_stamps']

                    if isinstance(pupil_timestamps, np.ndarray):
                        pupil_timestamps = pupil_timestamps.tolist()

                    _, _, _, _, pupil_sampling_rate = get_footer_info(stream)

            case 'GazeOrigin':

                gazeOrigin = stream['time_series']

                if isinstance(gazeOrigin, np.ndarray):
                    gazeOrigin = gazeOrigin.tolist()

                if gazeOrigin != []:

                    gazeOrigin_timestamps = stream['time_stamps']

                    if isinstance(gazeOrigin_timestamps, np.ndarray):
                        gazeOrigin_timestamps = gazeOrigin_timestamps.tolist()

                    _, _, _, _, gazeOrigin_sampling_rate = get_footer_info(stream)

            case 'Explore_84A1_ExG':

                eeg = stream['time_series']

                if isinstance(eeg, np.ndarray):
                    eeg = eeg.tolist()

                if eeg != []:

                    eeg_timestamps = stream['time_stamps']

                    if isinstance(eeg_timestamps, np.ndarray):
                        eeg_timestamps = eeg_timestamps.tolist()

                    _, _, _, _, eeg_sampling_rate = get_footer_info(stream)

                    channels_desc = streams[i]['info']['desc'][0]['channels'][0]['channel']
                    eeg_channels = [ch['name'][0] for ch in channels_desc]

            case 'GazeDirection':

                gazeDirection = stream['time_series']

                if isinstance(gazeDirection, np.ndarray):
                    gazeDirection = gazeDirection.tolist()

                if gazeDirection != []:

                    gazeDirection_timestamps = stream['time_stamps']

                    if isinstance(gazeDirection_timestamps, np.ndarray):
                        gazeDirection_timestamps = gazeDirection_timestamps.tolist()

                    _, _, _, _, gazeDirection_sampling_rate = get_footer_info(stream)

    physio_data = { 'Gaze2D': {'Info': ['x', 'y'], 'Value' : gaze2D, 'TimeStamp' : gaze2D_timestamps, 'SamplingRate' : gaze2D_sampling_rate},
                    'Gaze3D': {'Info': ['x', 'y', 'z'], 'Value' : gaze3D, 'TimeStamp' : gaze3D_timestamps, 'SamplingRate' : gaze3D_sampling_rate},
                    'PupilDiameter': {'Info': ['left', 'right'], 'Value' : pupil, 'TimeStamp' : pupil_timestamps, 'SamplingRate' : pupil_sampling_rate},
                    'GazeOrigin': {'Info': ['left_x', 'left_y', 'left_z', 'right_x', 'right_y', 'right_z'], 'Value' : gazeOrigin, 'TimeStamp' : gazeOrigin_timestamps, 'SamplingRate' : gazeOrigin_sampling_rate},
                    'GazeDirection': {'Info': ['left_x', 'left_y', 'left_z', 'right_x', 'right_y', 'right_z'], 'Value' : gazeDirection, 'TimeStamp' : gazeDirection_timestamps, 'SamplingRate' : gazeDirection_sampling_rate},
                    'Marker': {'Info': ['event'], 'Value' : marker, 'TimeStamp' : marker_timestamps, 'SamplingRate' : marker_sampling_rate},
                    'EEG': {'Info': eeg_channels, 'Value' : eeg, 'TimeStamp' : eeg_timestamps, 'SamplingRate' : eeg_sampling_rate}
                    }

    physio_data = convert_ndarray_to_list(physio_data)

    return physio_data


In [5]:
def get_physio_data_webcam_dry(file_path):

    streams, file_header = pyxdf.load_xdf(file_path, verbose=False)

    # Initialize variables
    gaze2D, gaze2D_timestamps = [], []
    gaze2D_sampling_rate = 0

    marker, eeg = [], []
    marker_timestamps, eeg_timestamps = [], []
    marker_sampling_rate, eeg_sampling_rate = 0, 0

    eeg_channels = []

    for i, stream in enumerate(streams):

        stream_name = stream['info']['name'][0]

        match stream_name:

            case 'GazeWebcam':

                gaze2D = stream['time_series']

                if isinstance(gaze2D, np.ndarray):
                    gaze2D = gaze2D.tolist()

                if gaze2D != []:

                    gaze2D_timestamps = stream['time_stamps']

                    if isinstance(gaze2D_timestamps, np.ndarray):
                        gaze2D_timestamps = gaze2D_timestamps.tolist()

                    _, _, _, _, gaze2D_sampling_rate = get_footer_info(stream)

            case 'Explore_84A1_ExG':

                eeg = stream['time_series']

                if isinstance(eeg, np.ndarray):
                    eeg = eeg.tolist()

                if eeg != []:

                    eeg_timestamps = stream['time_stamps']

                    if isinstance(eeg_timestamps, np.ndarray):
                        eeg_timestamps = eeg_timestamps.tolist()

                    _, _, _, _, eeg_sampling_rate = get_footer_info(stream)

                    channels_desc = streams[i]['info']['desc'][0]['channels'][0]['channel']
                    eeg_channels = [ch['name'][0] for ch in channels_desc]

            case 'Explore_84A1_Marker':

                marker = stream['time_series']

                if isinstance(marker, np.ndarray):
                    marker = marker.tolist()

                if marker != []:

                    marker_timestamps = stream['time_stamps']

                    if isinstance(marker_timestamps, np.ndarray):
                        marker_timestamps = marker_timestamps.tolist()

                    _, _, _, _, marker_sampling_rate = get_footer_info(stream)

    physio_data = { 'Gaze2D': {'Info': ['x', 'y'], 'Value' : gaze2D, 'TimeStamp' : gaze2D_timestamps, 'SamplingRate' : gaze2D_sampling_rate},
                    'Marker': {'Info': ['event'], 'Value' : marker, 'TimeStamp' : marker_timestamps, 'SamplingRate' : marker_sampling_rate},
                    'EEG': {'Info': eeg_channels, 'Value' : eeg, 'TimeStamp' : eeg_timestamps, 'SamplingRate' : eeg_sampling_rate}
                    }

    physio_data = convert_ndarray_to_list(physio_data)

    return physio_data

## README: Change File Name Here and Run All Previous Sections

* use 'get_physio_data_webcam_dry()' for OpenBCI and webcam experiment
* use 'get_physio_tobbi_gel()' for Tobbi Glasses with Gel electrode experiment

In [9]:
file_name = 'exp5'
data_path = f'experiment_data/exp5/'
database_path = os.path.join(data_path, file_name + '.xdf')

if os.path.exists(database_path):

    print(f"database path exists. getting physio data from the file : {file_name}")

    # physio_data_dic = get_physio_tobbi_gel(database_path)
    physio_data_dic = get_physio_data_webcam_dry(database_path)

    with open(os.path.join(data_path, f'{file_name}.json'), 'w') as json_file:
        json.dump(physio_data_dic, json_file, indent = 4)
    print("completed dictionary. Exported")

else:
    print(f"couldn't find the file: {database_path}")

database path exists. getting physio data from the file : exp5
completed dictionary. Exported


## Post Data Extraction Investigation

Use if needed for data inspections

In [ ]:
data_file = 'exp3.json'

with open(data_file, 'r') as json_file:
    json_data = json.load(json_file)

In [ ]:
import pandas as pd

def stream_to_dataframe(stream_data):

    values = stream_data['Value']
    info = stream_data.get('Info', [])

    data_dict = {
        "TIMESTAMP": stream_data['TimeStamp'],
        "SAMPLING_RATE": stream_data['SamplingRate']
    }

    for i, name in enumerate(info):
        try:
            data_dict[f"{name.upper()}"] = [value[i] for value in values]

        except IndexError:
            data_dict[f"{name.upper()}"] = values  # fallback for 1D data

    df = pd.DataFrame(data_dict)
    return df

In [ ]:
json_data.keys()

dict_keys(['Gaze2D', 'Marker', 'EEG'])

In [ ]:
gaze2D_df = stream_to_dataframe(json_data['Gaze2D'])
# gaze3D_df = stream_to_dataframe(json_data['Gaze3D'])
# pupil_df = stream_to_dataframe(json_data['PupilDiameter'])
# gazeOrigin_df = stream_to_dataframe(json_data['GazeOrigin'])
# gazeDirection_df = stream_to_dataframe(json_data['GazeDirection'])
marker_df = stream_to_dataframe(json_data['Marker'])
eeg_df = stream_to_dataframe(json_data['EEG'])

In [ ]:
gaze2D_df.head()

,TIMESTAMP,SAMPLING_RATE,X,Y
0,539943.694414,3,647.0,859.0
1,539964.171055,3,237.0,1044.0
2,539964.267969,3,510.0,661.0
3,539964.347558,3,442.0,527.0
4,539964.440361,3,357.0,516.0


In [ ]:
import pandas as pd

gaze2D_df = pd.DataFrame({

    "Timestamp": json_data['Gaze2D']['TimeStamp'],
    f"Value_{info[0].upper()}": [v[0] for v in json_data['Gaze2D']['Value']],
    f"Value_{info[1].upper()}": [v[1] for v in json_data['Gaze2D']['Value']],
    "SamplingRate": json_data['Gaze2D']['SamplingRate'],

})

gaze3D_df = pd.DataFrame({

    "Timestamp": json_data['Gaze3D']['TimeStamp'],
    "Value_X": [v[0] for v in json_data['Gaze3D']['Value']],
    "Value_Y": [v[1] for v in json_data['Gaze3D']['Value']],
    "Value_Z": [v[2] for v in json_data['Gaze3D']['Value']],
    "SamplingRate": json_data['Gaze3D']['SamplingRate'],

})

pupil_df = pd.DataFrame({

    "Timestamp": json_data['PupilDiameter']['TimeStamp'],
    "Left": [v[0] for v in json_data['PupilDiameter']['Value']],
    "Right": [v[1] for v in json_data['PupilDiameter']['Value']],
    "SamplingRate": json_data['PupilDiameter']['SamplingRate'],

})

gazeOrigin_df = pd.DataFrame({

    "Timestamp": json_data['GazeOrigin']['TimeStamp'],
    "Left_X": [v[0] for v in json_data['GazeOrigin']['Value']],
    "Left_Y": [v[1] for v in json_data['GazeOrigin']['Value']],
    "Left_Z": [v[2] for v in json_data['GazeOrigin']['Value']],
    "Right_X": [v[3] for v in json_data['GazeOrigin']['Value']],
    "Right_Y": [v[4] for v in json_data['GazeOrigin']['Value']],
    "Right_Z": [v[5] for v in json_data['GazeOrigin']['Value']],
    "SamplingRate": json_data['GazeOrigin']['SamplingRate'],

})

gazeDirection_df = pd.DataFrame({

    "Timestamp": json_data['GazeDirection']['TimeStamp'],
    "Left_X": [v[0] for v in json_data['GazeDirection']['Value']],
    "Left_Y": [v[1] for v in json_data['GazeDirection']['Value']],
    "Left_Z": [v[2] for v in json_data['GazeDirection']['Value']],
    "Right_X": [v[3] for v in json_data['GazeDirection']['Value']],
    "Right_Y": [v[4] for v in json_data['GazeDirection']['Value']],
    "Right_Z": [v[5] for v in json_data['GazeDirection']['Value']],
    "SamplingRate": json_data['GazeDirection']['SamplingRate'],

})

marker_df = pd.DataFrame({

    "Timestamp": json_data['Marker']['TimeStamp'],
    "Marker": json_data['Marker']['Value'],
    "SamplingRate": json_data['Marker']['SamplingRate'],

})

eeg_df = pd.DataFrame({

    f"json_data['EEG']['Info'][0]" :
})

print(gaze2D_df)

           Timestamp  Value_X  Value_Y
0       42066.876409   0.3857   0.2740
1       42066.886427   0.3875   0.3009
2       42066.896445   0.3981   0.3196
3       42066.906464   0.3990   0.3335
4       42066.916482   0.4021   0.3541
...              ...      ...      ...
237902  44450.257133   0.4110   0.2948
237903  44450.267151   0.4110   0.3004
237904  44450.277169   0.4109   0.2968
237905  44450.287188   0.4107   0.2989
237906  44450.297206   0.4105   0.3002

[237907 rows x 3 columns]


## Inspection

In [ ]:
channels_desc = streams[6]['info']['desc'][0]['channels'][0]['channel']
eeg_info = [ch['name'][0] for ch in channels_desc]

print(type(eeg_info))

<class 'list'>


In [ ]:
# print(streams[1]['time_series'])
# print()
print(streams[1]['time_stamps'])
print()
print(streams[1].keys())
print()
print(streams[1]['info'].keys())
print()
print(streams[1]['footer'].keys())
print()
print(streams[1]['footer']['info']['first_timestamp'])

[42066.87640854 42066.88642687 42066.8964452  ... 44450.27716933
 44450.28718766 44450.29720599]

dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values'])

dict_keys(['name', 'type', 'channel_count', 'channel_format', 'source_id', 'nominal_srate', 'version', 'created_at', 'uid', 'session_id', 'hostname', 'v4address', 'v4data_port', 'v4service_port', 'v6address', 'v6data_port', 'v6service_port', 'desc', 'stream_id', 'effective_srate', 'segments', 'first_timestamp'])

dict_keys(['info'])

['42066.8717138']


In [ ]:
print(streams[5]['time_series'])
print()
print(streams[5]['time_stamps'])
print()
print(streams[5]['info']['first_timestamp'])
print()
print(streams[5]['footer']['info']['first_timestamp'])

[[ 35.8   -8.57 -27.07 -32.09  -8.96 -27.27]
 [ 35.76  -8.58 -27.09 -32.06  -9.17 -27.24]
 [ 35.59  -8.62 -27.06 -32.07  -9.26 -27.22]
 ...
 [ 34.84  -7.46 -27.18 -31.5   -7.49 -27.46]
 [ 34.84  -7.47 -27.18 -31.5   -7.49 -27.46]
 [ 34.84  -7.48 -27.18 -31.5   -7.49 -27.46]]

[42066.87641138 42066.88642971 42066.89644804 ... 44450.27716738
 44450.28718571 44450.29720404]

[]

['42066.8717127']
